In [1]:
!git clone https://github.com/Ethereal1z/DRMGNE.git

fatal: destination path 'DRMGNE' already exists and is not an empty directory.


In [2]:
import pandas as pd, os
from IPython.display import display

DATA_DIR = '/content/DRMGNE/data/B-dataset'
files_to_check = [
    "DiseaseGIP.csv","DiseasePS.csv","DrugDiseaseAssociationNumber.csv",
    "DrugFingerprint.csv","DrugGIP.csv","DrugProteinAssociationNumber.csv",
    "ProteinDiseaseAssociationNumber.csv","Protein_sequence.csv","adj.csv",
]
for fn in files_to_check:
    fp = os.path.join(DATA_DIR, fn)
    if os.path.exists(fp):
        df = pd.read_csv(fp, header=None)
        print(f"{fn}: {df.shape}")
    else:
        print(f"MISSING: {fn}")

DiseaseGIP.csv: (599, 599)
DiseasePS.csv: (599, 599)
DrugDiseaseAssociationNumber.csv: (18411, 2)
DrugFingerprint.csv: (270, 270)
DrugGIP.csv: (270, 270)
DrugProteinAssociationNumber.csv: (3111, 2)
ProteinDiseaseAssociationNumber.csv: (5899, 2)
Protein_sequence.csv: (1022, 1022)
adj.csv: (270, 599)


In [3]:
from dataclasses import dataclass, field
from typing import Dict, List

N_DRUG, N_DIS, N_PROT = 269, 598, 1021
N_TOTAL = N_DRUG + N_DIS + N_PROT  # 1888

@dataclass
class Config:
    # data
    data_dir: str = "/content/DRMGNE/data/B-dataset"
    n_drug: int = N_DRUG
    n_dis:  int = N_DIS
    n_prot: int = N_PROT
    sim_fusion: str = "gip_fill"
    recompute_gip: bool = False
    sim_topk: int = 50
    sim_threshold: float = 0.0
    metapaths: Dict[int, List[str]] = field(default_factory=lambda: {
        1: ["RR","DD","PP","RD","RP","PD"],
        2: ["RPR","RDR","DPD","RPD","RDP"],
        3: ["RPDR","RDPD","RPRP"],
    })
    max_distance: int = 3
    binarize_adj: bool = True
    # model
    emb_dim: int = 128
    n_layers: int = 1
    dropout: float = 0.2
    residual: bool = True
    layernorm: bool = True
    node_attn: str = "gat"
    predictor_hidden: int = 128
    # training
    neg_ratio: int = 1
    resample_neg_each_epoch: bool = True
    lr: float = 1e-3
    weight_decay: float = 5e-4
    epochs: int = 250
    early_stop_patience: int = 250
    seed: int = 42
    n_folds: int = 5
    eval_mode: str = "balanced"
    score_threshold: float = 0.5
    # ver2: topk meta-paths
    metapath_topk: int = 10
    # ver3 new levers
    hard_neg_weight: float = 0.5   # EXP-B/C bidirectional hard neg weight
    sim_cl_weight:   float = 0.0   # EXP-D similarity contrastive loss (0=off)
    sim_cl_temp:     float = 0.05  # EXP-D InfoNCE temperature
    gt_layers: int = 2             # EXP-E/F graph transformer layers
    gt_heads:  int = 4             # EXP-E/F graph transformer heads

print("Config defined. ver3 defaults: emb_dim=128, gat, topk=10, epochs=250")

Config defined. ver3 defaults: emb_dim=128, gat, topk=10, epochs=250


In [4]:
import os, numpy as np, pandas as pd

def _read_matrix(path, shape=None):
    df = pd.read_csv(path, header=0, index_col=0)
    arr = df.apply(pd.to_numeric, errors="coerce").to_numpy(np.float64)
    arr = np.nan_to_num(arr, nan=0.0).astype(np.float32)
    if shape is not None:
        assert arr.shape == shape, f"{path}: got {arr.shape}, expected {shape}"
    return arr

def _read_pairs(path, max_col0, max_col1):
    df = pd.read_csv(path, header=0)
    pairs = df.to_numpy(dtype=np.int64)
    if pairs.min() >= 1:
        pairs = pairs - 1
    if pairs[:, 0].max() >= max_col0 and pairs[:, 1].max() < max_col0:
        pairs = pairs[:, [1, 0]]
    assert pairs[:, 0].max() < max_col0 and pairs[:, 1].max() < max_col1
    return pairs

def _topk_sparsify(S, k, thr):
    """Keep top-k off-diagonal neighbours per row above thr; symmetrise. Square only."""
    S = S.copy(); np.fill_diagonal(S, 0.0)
    if thr > 0: S[S <= thr] = 0.0
    if k and k > 0:
        for i in range(S.shape[0]):
            row = S[i]
            if (row > 0).sum() > k:
                kth = np.partition(row, -k)[-k]
                row[row < kth] = 0.0
    return np.maximum(S, S.T)

def _fuse(sources, mode):
    sources = [s for s in sources if s is not None]
    if mode == "struct_only": return sources[0]
    if mode == "gip_only":    return sources[-1]
    if mode == "gip_fill":
        base = sources[0].copy(); gip = sources[-1]
        base[base <= 0] = gip[base <= 0]; return base
    return np.mean(np.stack(sources, 0), axis=0).astype(np.float32)

def gip_kernel(profiles):
    sq = (profiles ** 2).sum(1)
    gamma = 1.0 / (sq.mean() + 1e-12)
    G = sq[:, None] + sq[None, :] - 2.0 * profiles @ profiles.T
    np.clip(G, 0, None, out=G)
    return np.exp(-gamma * G).astype(np.float32)

class BDataset:
    def __init__(self, cfg):
        self.cfg = cfg
        d = cfg.data_dir
        nR, nD, nP = cfg.n_drug, cfg.n_dis, cfg.n_prot
        self.A_rd = (_read_matrix(os.path.join(d,"adj.csv"), (nR,nD)) > 0).astype(np.float32)
        rp = _read_pairs(os.path.join(d,"DrugProteinAssociationNumber.csv"), nR, nP)
        self.A_rp = np.zeros((nR,nP), np.float32); self.A_rp[rp[:,0], rp[:,1]] = 1.0
        pd_ = _read_pairs(os.path.join(d,"ProteinDiseaseAssociationNumber.csv"), nD, nP)
        self.A_pd = np.zeros((nP,nD), np.float32); self.A_pd[pd_[:,1], pd_[:,0]] = 1.0
        self._drug_struct = _read_matrix(os.path.join(d,"DrugFingerprint.csv"), (nR,nR))
        self._drug_gip    = _read_matrix(os.path.join(d,"DrugGIP.csv"),         (nR,nR))
        self._dis_struct  = _read_matrix(os.path.join(d,"DiseasePS.csv"),        (nD,nD))
        self._dis_gip     = _read_matrix(os.path.join(d,"DiseaseGIP.csv"),       (nD,nD))
        self._prot_seq    = _read_matrix(os.path.join(d,"Protein_sequence.csv"), (nP,nP))
        gd = _read_matrix(os.path.join(d,"ProteinGIP_Drug.csv"),    (nP,nP))
        gdi= _read_matrix(os.path.join(d,"ProteinGIP_Disease.csv"), (nP,nP))
        self._prot_gip = ((gd + gdi) / 2.0).astype(np.float32)

    def build_similarities(self, A_rd_train):
        cfg = self.cfg
        drug_gip = gip_kernel(A_rd_train)    if cfg.recompute_gip else self._drug_gip
        dis_gip  = gip_kernel(A_rd_train.T)  if cfg.recompute_gip else self._dis_gip
        S_rr = _topk_sparsify(_fuse([self._drug_struct, drug_gip], cfg.sim_fusion), cfg.sim_topk, cfg.sim_threshold)
        S_dd = _topk_sparsify(_fuse([self._dis_struct,  dis_gip],  cfg.sim_fusion), cfg.sim_topk, cfg.sim_threshold)
        S_pp = _topk_sparsify(_fuse([self._prot_seq, self._prot_gip], cfg.sim_fusion), cfg.sim_topk, cfg.sim_threshold)
        return S_rr, S_dd, S_pp

    @property
    def positives(self):
        r, c = np.where(self.A_rd > 0); return np.stack([r, c], 1)

def sample_negatives(pos_set, n_neg, nR, nD, rng, forbid=None):
    forbid = forbid or set(); out = []
    while len(out) < n_neg:
        r = rng.integers(0, nR, size=n_neg); d = rng.integers(0, nD, size=n_neg)
        for a, b in zip(r, d):
            key = int(a)*nD + int(b)
            if key in pos_set or key in forbid: continue
            out.append((a, b))
            if len(out) >= n_neg: break
    return np.array(out, dtype=np.int64)

print("BDataset, gip_kernel, _topk_sparsify, sample_negatives defined.")

BDataset, gip_kernel, _topk_sparsify, sample_negatives defined.


In [5]:
def _base_blocks(A_rd, A_rp, A_pd, S_rr, S_dd, S_pp):
    return {"RR":S_rr,"DD":S_dd,"PP":S_pp,
            "RD":A_rd,"DR":A_rd.T,"RP":A_rp,"PR":A_rp.T,"PD":A_pd,"DP":A_pd.T}

def _metapath_commuting(path, blocks):
    mats = [blocks[path[i]+path[i+1]] for i in range(len(path)-1)]
    M = mats[0]
    for m in mats[1:]: M = M @ m
    return M

def build_H0(A_rd, A_rp, A_pd, S_rr, S_dd, S_pp):
    nR,nD,nP = A_rd.shape[0],A_rd.shape[1],A_rp.shape[1]; N=nR+nD+nP
    H0 = np.zeros((N,N), np.float32)
    H0[0:nR,0:nR]=S_rr;             H0[0:nR,nR:nR+nD]=A_rd;      H0[0:nR,nR+nD:N]=A_rp
    H0[nR:nR+nD,0:nR]=A_rd.T;       H0[nR:nR+nD,nR:nR+nD]=S_dd;  H0[nR:nR+nD,nR+nD:N]=A_pd.T
    H0[nR+nD:N,0:nR]=A_rp.T;        H0[nR+nD:N,nR:nR+nD]=A_pd;   H0[nR+nD:N,nR+nD:N]=S_pp
    return H0

def build_H0_decoupled(A_rd, A_rp, A_pd, S_rr, S_dd, S_pp):
    """Similarity-only diagonal H0: associations enter ONLY as meta-path edges."""
    nR,nD,nP = A_rd.shape[0],A_rd.shape[1],A_rp.shape[1]; N=nR+nD+nP
    H0 = np.zeros((N,N), np.float32)
    H0[0:nR,0:nR]=S_rr; H0[nR:nR+nD,nR:nR+nD]=S_dd; H0[nR+nD:N,nR+nD:N]=S_pp
    return H0

def build_metapath_adjs(cfg, A_rd, A_rp, A_pd, S_rr, S_dd, S_pp):
    """Returns {distance: [(name, NxN adj)]}. Row-wise topk for rectangular paths."""
    nR,nD,nP = cfg.n_drug,cfg.n_dis,cfg.n_prot; N=nR+nD+nP
    off={"R":0,"D":nR,"P":nR+nD}; cnt={"R":nR,"D":nD,"P":nP}
    blocks = _base_blocks(A_rd, A_rp, A_pd, S_rr, S_dd, S_pp)
    topk = getattr(cfg, "metapath_topk", 0)
    out = {}
    for dist, names in cfg.metapaths.items():
        if dist > cfg.max_distance: continue
        lst = []
        for name in names:
            M = _metapath_commuting(name, blocks)
            if topk > 0:
                M = M.copy()
                for _i in range(M.shape[0]):
                    _row = M[_i]
                    if (_row > 0).sum() > topk:
                        _kth = np.partition(_row, -topk)[-topk]
                        M[_i, M[_i] < _kth] = 0.0
            s_type, e_type = name[0], name[-1]
            A = np.zeros((N,N), np.float32)
            rs = slice(off[s_type], off[s_type]+cnt[s_type])
            cs = slice(off[e_type], off[e_type]+cnt[e_type])
            A[rs,cs] = M; A = np.maximum(A, A.T)
            if cfg.binarize_adj: A = (A > 0).astype(np.float32)
            else:
                deg = A.sum(1, keepdims=True); deg[deg==0]=1.0; A = A/deg
            np.fill_diagonal(A, 1.0)
            lst.append((name, A))
        out[dist] = lst
    return out

def mask_test_edges(A_rd, test_pairs):
    A = A_rd.copy(); A[test_pairs[:,0], test_pairs[:,1]] = 0.0; return A

print("build_H0, build_H0_decoupled, build_metapath_adjs, mask_test_edges defined.")

build_H0, build_H0_decoupled, build_metapath_adjs, mask_test_edges defined.


In [6]:
import torch, torch.nn as nn, torch.nn.functional as F

NEG_INF = -9e15

class MPHAMLayer(nn.Module):
    def __init__(self, in_dim, out_dim, n_distances, cfg):
        super().__init__()
        self.cfg = cfg
        self.node_W = nn.Linear(in_dim, out_dim, bias=False)
        self.a_src  = nn.Parameter(torch.empty(out_dim))
        self.a_dst  = nn.Parameter(torch.empty(out_dim))
        self.pat_lin  = nn.Linear(out_dim, out_dim)
        self.u_a      = nn.Parameter(torch.empty(out_dim))
        self.dist_lin = nn.Linear(out_dim, out_dim)
        self.u_b      = nn.Parameter(torch.empty(out_dim))
        self.drop = nn.Dropout(cfg.dropout)
        self.norm = nn.LayerNorm(out_dim) if cfg.layernorm else nn.Identity()
        self.res_proj = (nn.Linear(in_dim, out_dim, bias=False)
                         if (cfg.residual and in_dim != out_dim) else None)
        nn.init.xavier_uniform_(self.node_W.weight)
        nn.init.xavier_uniform_(self.pat_lin.weight)
        nn.init.xavier_uniform_(self.dist_lin.weight)
        for p in (self.a_src, self.a_dst, self.u_a, self.u_b):
            nn.init.normal_(p, std=0.1)
        if self.res_proj is not None:
            nn.init.xavier_uniform_(self.res_proj.weight)

    def _node_attention(self, Wh, adj):
        if self.cfg.node_attn == "paper":
            scores = (Wh @ self.a_dst).unsqueeze(0).expand(adj.size(0), -1)
        else:
            e_src = Wh @ self.a_src; e_dst = Wh @ self.a_dst
            scores = F.leaky_relu(e_src.unsqueeze(1) + e_dst.unsqueeze(0), 0.2)
        scores = scores.masked_fill(adj == 0, NEG_INF)
        alpha  = torch.softmax(scores, dim=1)
        alpha  = torch.where((adj.sum(1, keepdim=True)==0), torch.zeros_like(alpha), alpha)
        return alpha @ Wh

    def forward(self, H, adjs):
        Wh = self.drop(F.relu(self.node_W(H)))
        Hk_list = []
        for dist in sorted(adjs.keys()):
            Hkm = [self._node_attention(Wh, adj) for _, adj in adjs[dist]]
            U   = [torch.tanh(self.pat_lin(h)) for h in Hkm]
            s   = torch.stack([u @ self.u_a for u in U], dim=1)
            beta = torch.softmax(s, dim=1)
            Hk  = sum(beta[:, m:m+1] * Hkm[m] for m in range(len(Hkm)))
            Hk_list.append(Hk)
        Ud = [torch.tanh(self.dist_lin(h)) for h in Hk_list]
        sd = torch.stack([u @ self.u_b for u in Ud], dim=1)
        gamma = torch.softmax(sd, dim=1)
        out = sum(gamma[:, k:k+1] * Hk_list[k] for k in range(len(Hk_list)))
        if self.cfg.residual:
            res = H if self.res_proj is None else self.res_proj(H)
            out = out + res
        return self.norm(out)

class MPHAM(nn.Module):
    def __init__(self, cfg, in_dim, n_distances):
        super().__init__(); self.cfg = cfg
        dims = [in_dim] + [cfg.emb_dim] * cfg.n_layers
        self.layers = nn.ModuleList([
            MPHAMLayer(dims[i], dims[i+1], n_distances, cfg) for i in range(cfg.n_layers)])
        self.predictor = nn.Sequential(
            nn.Linear(2*cfg.emb_dim, cfg.predictor_hidden), nn.ReLU(),
            nn.Dropout(cfg.dropout), nn.Linear(cfg.predictor_hidden, 1))

    def encode(self, H0, adjs):
        H = H0
        for layer in self.layers: H = layer(H, adjs)
        return H

    def score(self, H, drug_idx, dis_idx):
        hr = H[drug_idx]; hd = H[self.cfg.n_drug + dis_idx]
        return self.predictor(torch.cat([hr, hd], dim=1)).squeeze(-1)

class MPHAMv2(MPHAM):
    """Element-wise product (DistMult/DRMGNE) decoder."""
    def __init__(self, cfg, in_dim, n_distances):
        super().__init__(cfg, in_dim, n_distances)
        d = cfg.emb_dim
        self.predictor = nn.Sequential(
            nn.Linear(d, cfg.predictor_hidden), nn.ReLU(),
            nn.Dropout(cfg.dropout), nn.Linear(cfg.predictor_hidden, 1))

    def score_direct(self, h_d, h_di):
        return self.predictor(h_d * h_di).squeeze(-1)

    def score(self, H, drug_idx, dis_idx):
        h_d  = H[drug_idx]; h_di = H[self.cfg.n_drug + dis_idx]
        return self.score_direct(h_d, h_di)

print("MPHAMLayer, MPHAM, MPHAMv2 (with score_direct) defined.")

MPHAMLayer, MPHAM, MPHAMv2 (with score_direct) defined.


In [7]:
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score

def _adjs_to_torch(adjs, device):
    return {k: [(n, torch.from_numpy(a).to(device)) for n, a in v] for k, v in adjs.items()}

def _metrics(y, prob, thr=0.5):
    pred = (prob >= thr).astype(int)
    return (roc_auc_score(y, prob), average_precision_score(y, prob),
            precision_score(y, pred, zero_division=0),
            recall_score(y, pred, zero_division=0),
            f1_score(y, pred, zero_division=0))

@torch.no_grad()
def evaluate(cfg, model, H0, adjs, test_pos, test_neg, device):
    model.eval()
    H = model.encode(H0, adjs)
    pairs = np.concatenate([test_pos, test_neg], 0)
    y = np.concatenate([np.ones(len(test_pos)), np.zeros(len(test_neg))])
    di = torch.from_numpy(pairs[:,0]).to(device)
    si = torch.from_numpy(pairs[:,1]).to(device)
    prob = torch.sigmoid(model.score(H, di, si)).cpu().numpy()
    return _metrics(y, prob, cfg.score_threshold)

def _agg(rows, tag):
    a = np.array(rows); m, s = a.mean(0), a.std(0)
    print(f"  [{tag}] AUC={m[0]:.3f}+/-{s[0]:.3f}  AUPR={m[1]:.3f}+/-{s[1]:.3f}")
    return m, s

print("evaluate, _metrics, _adjs_to_torch, _agg defined.")

evaluate, _metrics, _adjs_to_torch, _agg defined.


In [8]:
def rotate_operator(a, b):
    """RotatE: complex multiplication. Requires even emb_dim.
    Source: GCGB/model.py:14-20."""
    a_re, a_im = a.chunk(2, dim=-1)
    b_re, b_im = b.chunk(2, dim=-1)
    return torch.cat([a_re*b_re - a_im*b_im,
                      a_re*b_im + a_im*b_re], dim=-1)

class MPHAMv5(MPHAMv2):
    """EXP-A: RotatE + mul 4-way decoder (GCGB-style).
    Input to predictor: cat([h_d, h_di, h_d*h_di, rotate(h_d,h_di)]) = 4*emb_dim.
    Captures individual features, DistMult interaction, and asymmetric RotatE relation.
    Requires emb_dim even (128, 256 both fine).
    """
    def __init__(self, cfg, in_dim, n_distances):
        super().__init__(cfg, in_dim, n_distances)
        d = cfg.emb_dim
        self.predictor = nn.Sequential(
            nn.Linear(4*d, cfg.predictor_hidden), nn.ReLU(),
            nn.Dropout(cfg.dropout), nn.Linear(cfg.predictor_hidden, 1))

    def score_direct(self, h_d, h_di):
        mul = h_d * h_di; rot = rotate_operator(h_d, h_di)
        return self.predictor(torch.cat([h_d, h_di, mul, rot], dim=-1)).squeeze(-1)

    def score(self, H, drug_idx, dis_idx):
        return self.score_direct(H[drug_idx], H[self.cfg.n_drug + dis_idx])

print("rotate_operator, MPHAMv5 defined. Predictor input = 4*emb_dim =", 4*128, "for default cfg.")

rotate_operator, MPHAMv5 defined. Predictor input = 4*emb_dim = 512 for default cfg.


In [9]:
def run_fold_v5(cfg, ds, train_pos, test_pos, device, rng, val_frac=0.1):
    """EXP-C: MPHAMv5 (RotatE+mul) + decoupled H0 + bidirectional hard negatives.

    Hard-negative strategy (DRMGNE both directions):
      score_r = mean(h_drug_pos * h_drug_neg).detach()
      h_drug_hard = score_r*h_drug_pos + (1-score_r)*h_drug_neg
      score_d = mean(h_dis_pos  * h_dis_neg).detach()
      h_dis_hard  = score_d*h_dis_pos  + (1-score_d)*h_dis_neg
      loss_hard = BCE(score_direct(h_drug_hard, h_dis_hard), zeros)
    """
    nR, nD = cfg.n_drug, cfg.n_dis
    pos_set = set(map(int, ds.positives[:,0]*nD + ds.positives[:,1]))
    n_val = max(1, int(len(train_pos)*val_frac))
    vidx  = rng.choice(len(train_pos), n_val, replace=False)
    vmask = np.ones(len(train_pos), bool); vmask[vidx] = False
    val_pos = train_pos[vidx]; tr_pos = train_pos[vmask]

    A_rd_tr = mask_test_edges(ds.A_rd, test_pos)
    S_rr, S_dd, S_pp = ds.build_similarities(A_rd_tr)
    H0_np   = build_H0_decoupled(A_rd_tr, ds.A_rp, ds.A_pd, S_rr, S_dd, S_pp)
    adjs_np = build_metapath_adjs(cfg, A_rd_tr, ds.A_rp, ds.A_pd, S_rr, S_dd, S_pp)
    H0   = torch.from_numpy(H0_np).to(device)
    adjs = _adjs_to_torch(adjs_np, device)

    model = MPHAMv5(cfg, in_dim=H0.shape[1], n_distances=len(adjs)).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    val_neg   = sample_negatives(pos_set, len(val_pos),      nR, nD, rng)
    test_neg1 = sample_negatives(pos_set, len(test_pos),     nR, nD, rng)
    test_neg5 = sample_negatives(pos_set, len(test_pos)*5,   nR, nD, rng)

    best_val_auc=-1; best_test_auc=-1; patience=0
    best_state_val  = {k:v.clone() for k,v in model.state_dict().items()}
    best_state_test = {k:v.clone() for k,v in model.state_dict().items()}

    for epoch in range(cfg.epochs):
        model.train()
        if epoch==0 or cfg.resample_neg_each_epoch:
            tr_neg = sample_negatives(pos_set, len(tr_pos)*cfg.neg_ratio, nR, nD, rng)
        pos_di = torch.from_numpy(tr_pos[:,0]).to(device)
        pos_si = torch.from_numpy(tr_pos[:,1]).to(device)
        neg_di = torch.from_numpy(tr_neg[:,0]).to(device)
        neg_si = torch.from_numpy(tr_neg[:,1]).to(device)

        opt.zero_grad()
        H = model.encode(H0, adjs)
        h_drug_pos = H[pos_di];          h_dis_pos = H[nR + pos_si]
        h_drug_neg = H[neg_di];          h_dis_neg = H[nR + neg_si]

        # base BCE
        loss = F.binary_cross_entropy_with_logits(
            torch.cat([model.score_direct(h_drug_pos, h_dis_pos),
                       model.score_direct(h_drug_neg, h_dis_neg)]),
            torch.cat([torch.ones(len(pos_di), device=device),
                       torch.zeros(len(neg_di), device=device)]))

        # bidirectional hard negatives (DRMGNE, both drug and disease direction)
        n_mix = min(len(h_drug_pos), len(h_drug_neg))
        sr = torch.mean(h_drug_pos[:n_mix] * h_drug_neg[:n_mix]).detach()
        h_drug_hard = sr * h_drug_pos[:n_mix] + (1.0-sr) * h_drug_neg[:n_mix]
        sd = torch.mean(h_dis_pos[:n_mix]  * h_dis_neg[:n_mix]).detach()
        h_dis_hard  = sd * h_dis_pos[:n_mix]  + (1.0-sd) * h_dis_neg[:n_mix]
        loss_hard = F.binary_cross_entropy_with_logits(
            model.score_direct(h_drug_hard, h_dis_hard),
            torch.zeros(n_mix, device=device))
        loss = loss + cfg.hard_neg_weight * loss_hard

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()

        vauc,*_ = evaluate(cfg, model, H0, adjs, val_pos, val_neg, device)
        if vauc > best_val_auc:
            best_val_auc=vauc; patience=0
            best_state_val = {k:v.clone() for k,v in model.state_dict().items()}
        else:
            patience += 1
            if patience >= cfg.early_stop_patience: break

        tauc,*_ = evaluate(cfg, model, H0, adjs, test_pos, test_neg1, device)
        if tauc > best_test_auc:
            best_test_auc=tauc
            best_state_test = {k:v.clone() for k,v in model.state_dict().items()}

    def _eval_both(state):
        model.load_state_dict(state)
        return (evaluate(cfg, model, H0, adjs, test_pos, test_neg1, device),
                evaluate(cfg, model, H0, adjs, test_pos, test_neg5, device))

    h_bal, h_imb = _eval_both(best_state_val)
    i_bal, i_imb = _eval_both(best_state_test)
    return h_bal, h_imb, i_bal, i_imb

def cross_validate_v5(cfg, val_frac=0.1):
    """EXP-C: RotatE+mul decoder + bidirectional hard negatives."""
    torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[EXP-C] device={device}  emb={cfg.emb_dim}  attn={cfg.node_attn}"
          f"  topk={cfg.metapath_topk}  hard_neg_w={cfg.hard_neg_weight}  ep={cfg.epochs}")
    ds=BDataset(cfg); pos=ds.positives
    rng=np.random.default_rng(cfg.seed); perm=rng.permutation(len(pos))
    folds=np.array_split(perm, cfg.n_folds)
    hb=[]; hi=[]; ib=[]; ii=[]
    for f in range(cfg.n_folds):
        tidx=folds[f]; trinx=np.concatenate([folds[g] for g in range(cfg.n_folds) if g!=f])
        h_bal,h_imb,i_bal,i_imb = run_fold_v5(cfg,ds,pos[trinx],pos[tidx],device,rng,val_frac)
        print(f"  fold {f}: HONEST AUC={h_bal[0]:.4f} AUPR1={h_bal[1]:.4f}"
              f" AUPR5={h_imb[1]:.4f}  | INFLATED AUC={i_bal[0]:.4f} AUPR1={i_bal[1]:.4f}")
        hb.append(h_bal); hi.append(h_imb); ib.append(i_bal); ii.append(i_imb)
    print(); hbm,hbs=_agg(hb,"HONEST   bal 1:1"); him,his=_agg(hi,"HONEST   imb 1:5")
    ibm,ibs=_agg(ib,"INFLATED bal 1:1");          _agg(ii,"INFLATED imb 1:5")
    return (hbm,hbs),(him,his),(ibm,ibs)

print("run_fold_v5 / cross_validate_v5 defined.")

run_fold_v5 / cross_validate_v5 defined.


In [11]:
print("="*65)
print("EXP-C: RotatE+mul decoder + Bidirectional Hard Negatives")
print("Base: decoupled H0, GAT, topk=10, emb=256, predictor_hidden=256")
print("="*65)
expc_results = cross_validate_v5(
    Config(node_attn="gat", emb_dim=256, predictor_hidden=256,
           metapath_topk=10, epochs=250, hard_neg_weight=0.5)
)

EXP-C: RotatE+mul decoder + Bidirectional Hard Negatives
Base: decoupled H0, GAT, topk=10, emb=256, predictor_hidden=256
[EXP-C] device=cuda  emb=256  attn=gat  topk=10  hard_neg_w=0.5  ep=250
  fold 0: HONEST AUC=0.9101 AUPR1=0.9246 AUPR5=0.7947  | INFLATED AUC=0.9101 AUPR1=0.9246
  fold 1: HONEST AUC=0.9011 AUPR1=0.9174 AUPR5=0.7860  | INFLATED AUC=0.9020 AUPR1=0.9180
  fold 2: HONEST AUC=0.9123 AUPR1=0.9280 AUPR5=0.8005  | INFLATED AUC=0.9131 AUPR1=0.9281
  fold 3: HONEST AUC=0.9127 AUPR1=0.9265 AUPR5=0.7918  | INFLATED AUC=0.9127 AUPR1=0.9265
  fold 4: HONEST AUC=0.9081 AUPR1=0.9217 AUPR5=0.7795  | INFLATED AUC=0.9082 AUPR1=0.9251

  [HONEST   bal 1:1] AUC=0.909+/-0.004  AUPR=0.924+/-0.004
  [HONEST   imb 1:5] AUC=0.908+/-0.004  AUPR=0.791+/-0.007
  [INFLATED bal 1:1] AUC=0.909+/-0.004  AUPR=0.924+/-0.003
  [INFLATED imb 1:5] AUC=0.908+/-0.004  AUPR=0.795+/-0.006


In [12]:
def sim_contrastive_loss(emb, sim_knn_np, temperature=0.05):
    """InfoNCE loss supervised by k-NN similarity matrix (label-free).
    Source: GCGB/contrastive_learning.py::similarity_contrastive (adapted).
    Positive pairs = k-NN neighbours; negative pairs = non-neighbours.
    No leakage: similarity from chemical/phenotype features, not associations.
    """
    N = emb.size(0)
    emb_n = F.normalize(emb, p=2, dim=1)
    score  = torch.mm(emb_n, emb_n.t())            # (N,N) cosine similarity
    sim_b  = torch.from_numpy((sim_knn_np > 0).astype(np.float32)).to(emb.device)
    pos_mask = sim_b - torch.eye(N, device=emb.device)  # exclude self
    neg_mask = (sim_b == 0).float(); neg_mask.fill_diagonal_(0.0)
    pos_score = (torch.exp(score / temperature) * pos_mask).sum(1)
    neg_score = (torch.exp(score / temperature) * neg_mask).sum(1)
    has_pos   = (pos_mask.sum(1) > 0)
    if not has_pos.any(): return emb.new_zeros(1).squeeze()
    return -torch.log(pos_score[has_pos] / (neg_score[has_pos] + 1e-8)).mean()

def run_fold_v6(cfg, ds, train_pos, test_pos, device, rng, val_frac=0.1):
    """EXP-D: EXP-C + similarity contrastive regularization.
    sim_cl_weight=0 -> identical to EXP-C (backward compatible).
    """
    nR, nD = cfg.n_drug, cfg.n_dis
    pos_set = set(map(int, ds.positives[:,0]*nD + ds.positives[:,1]))
    n_val = max(1, int(len(train_pos)*val_frac))
    vidx  = rng.choice(len(train_pos), n_val, replace=False)
    vmask = np.ones(len(train_pos), bool); vmask[vidx] = False
    val_pos = train_pos[vidx]; tr_pos = train_pos[vmask]

    A_rd_tr = mask_test_edges(ds.A_rd, test_pos)
    S_rr, S_dd, S_pp = ds.build_similarities(A_rd_tr)
    H0_np   = build_H0_decoupled(A_rd_tr, ds.A_rp, ds.A_pd, S_rr, S_dd, S_pp)
    adjs_np = build_metapath_adjs(cfg, A_rd_tr, ds.A_rp, ds.A_pd, S_rr, S_dd, S_pp)
    H0   = torch.from_numpy(H0_np).to(device)
    adjs = _adjs_to_torch(adjs_np, device)

    model = MPHAMv5(cfg, in_dim=H0.shape[1], n_distances=len(adjs)).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    val_neg   = sample_negatives(pos_set, len(val_pos),     nR, nD, rng)
    test_neg1 = sample_negatives(pos_set, len(test_pos),    nR, nD, rng)
    test_neg5 = sample_negatives(pos_set, len(test_pos)*5,  nR, nD, rng)
    best_val_auc=-1; best_test_auc=-1; patience=0
    best_state_val  = {k:v.clone() for k,v in model.state_dict().items()}
    best_state_test = {k:v.clone() for k,v in model.state_dict().items()}

    for epoch in range(cfg.epochs):
        model.train()
        if epoch==0 or cfg.resample_neg_each_epoch:
            tr_neg = sample_negatives(pos_set, len(tr_pos)*cfg.neg_ratio, nR, nD, rng)
        pos_di=torch.from_numpy(tr_pos[:,0]).to(device); pos_si=torch.from_numpy(tr_pos[:,1]).to(device)
        neg_di=torch.from_numpy(tr_neg[:,0]).to(device); neg_si=torch.from_numpy(tr_neg[:,1]).to(device)

        opt.zero_grad()
        H = model.encode(H0, adjs)
        h_drug_pos=H[pos_di]; h_dis_pos=H[nR+pos_si]
        h_drug_neg=H[neg_di]; h_dis_neg=H[nR+neg_si]

        loss = F.binary_cross_entropy_with_logits(
            torch.cat([model.score_direct(h_drug_pos,h_dis_pos),
                       model.score_direct(h_drug_neg,h_dis_neg)]),
            torch.cat([torch.ones(len(pos_di),device=device),
                       torch.zeros(len(neg_di),device=device)]))

        n_mix=min(len(h_drug_pos),len(h_drug_neg))
        sr=torch.mean(h_drug_pos[:n_mix]*h_drug_neg[:n_mix]).detach()
        h_drug_hard=sr*h_drug_pos[:n_mix]+(1-sr)*h_drug_neg[:n_mix]
        sd=torch.mean(h_dis_pos[:n_mix]*h_dis_neg[:n_mix]).detach()
        h_dis_hard=sd*h_dis_pos[:n_mix]+(1-sd)*h_dis_neg[:n_mix]
        loss = loss + cfg.hard_neg_weight * F.binary_cross_entropy_with_logits(
            model.score_direct(h_drug_hard, h_dis_hard),
            torch.zeros(n_mix, device=device))

        if cfg.sim_cl_weight > 0:
            h_all_drug = H[:nR]; h_all_dis = H[nR:nR+nD]
            loss = loss + cfg.sim_cl_weight * (
                sim_contrastive_loss(h_all_drug, S_rr, cfg.sim_cl_temp) +
                sim_contrastive_loss(h_all_dis,  S_dd, cfg.sim_cl_temp))

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()

        vauc,*_=evaluate(cfg,model,H0,adjs,val_pos,val_neg,device)
        if vauc>best_val_auc:
            best_val_auc=vauc; patience=0
            best_state_val={k:v.clone() for k,v in model.state_dict().items()}
        else:
            patience+=1
            if patience>=cfg.early_stop_patience: break
        tauc,*_=evaluate(cfg,model,H0,adjs,test_pos,test_neg1,device)
        if tauc>best_test_auc:
            best_test_auc=tauc
            best_state_test={k:v.clone() for k,v in model.state_dict().items()}

    def _eval_both(state):
        model.load_state_dict(state)
        return (evaluate(cfg,model,H0,adjs,test_pos,test_neg1,device),
                evaluate(cfg,model,H0,adjs,test_pos,test_neg5,device))

    h_bal,h_imb=_eval_both(best_state_val); i_bal,i_imb=_eval_both(best_state_test)
    return h_bal,h_imb,i_bal,i_imb

def cross_validate_v6(cfg, val_frac=0.1):
    """EXP-D: EXP-C + similarity contrastive loss."""
    torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
    device="cuda" if torch.cuda.is_available() else "cpu"
    print(f"[EXP-D] device={device}  emb={cfg.emb_dim}  attn={cfg.node_attn}"
          f"  topk={cfg.metapath_topk}  sim_cl_w={cfg.sim_cl_weight}"
          f"  temp={cfg.sim_cl_temp}  ep={cfg.epochs}")
    ds=BDataset(cfg); pos=ds.positives
    rng=np.random.default_rng(cfg.seed); perm=rng.permutation(len(pos))
    folds=np.array_split(perm,cfg.n_folds)
    hb=[]; hi=[]; ib=[]; ii=[]
    for f in range(cfg.n_folds):
        tidx=folds[f]; trinx=np.concatenate([folds[g] for g in range(cfg.n_folds) if g!=f])
        h_bal,h_imb,i_bal,i_imb=run_fold_v6(cfg,ds,pos[trinx],pos[tidx],device,rng,val_frac)
        print(f"  fold {f}: HONEST AUC={h_bal[0]:.4f} AUPR1={h_bal[1]:.4f}"
              f" AUPR5={h_imb[1]:.4f}  | INFLATED AUC={i_bal[0]:.4f}")
        hb.append(h_bal); hi.append(h_imb); ib.append(i_bal); ii.append(i_imb)
    print(); hbm,hbs=_agg(hb,"HONEST   bal 1:1"); him,his=_agg(hi,"HONEST   imb 1:5")
    ibm,ibs=_agg(ib,"INFLATED bal 1:1"); _agg(ii,"INFLATED imb 1:5")
    return (hbm,hbs),(him,his),(ibm,ibs)

print("sim_contrastive_loss, run_fold_v6, cross_validate_v6 defined.")

sim_contrastive_loss, run_fold_v6, cross_validate_v6 defined.


In [14]:
print("="*65)
print("EXP-D: EXP-C + Similarity Contrastive Loss (sim_cl_weight=1e-4)")
print("="*65)
expd_results = cross_validate_v6(
    Config(node_attn="gat", emb_dim=256, predictor_hidden=256,
           metapath_topk=10, epochs=250,
           hard_neg_weight=0.5, sim_cl_weight=1e-4, sim_cl_temp=0.05)
)

EXP-D: EXP-C + Similarity Contrastive Loss (sim_cl_weight=1e-4)
[EXP-D] device=cuda  emb=256  attn=gat  topk=10  sim_cl_w=0.0001  temp=0.05  ep=250
  fold 0: HONEST AUC=0.9095 AUPR1=0.9224 AUPR5=0.7858  | INFLATED AUC=0.9107
  fold 1: HONEST AUC=0.9034 AUPR1=0.9191 AUPR5=0.7896  | INFLATED AUC=0.9034
  fold 2: HONEST AUC=0.9112 AUPR1=0.9269 AUPR5=0.8012  | INFLATED AUC=0.9146
  fold 3: HONEST AUC=0.9109 AUPR1=0.9272 AUPR5=0.7990  | INFLATED AUC=0.9124
  fold 4: HONEST AUC=0.9074 AUPR1=0.9194 AUPR5=0.7734  | INFLATED AUC=0.9087

  [HONEST   bal 1:1] AUC=0.908+/-0.003  AUPR=0.923+/-0.004
  [HONEST   imb 1:5] AUC=0.908+/-0.003  AUPR=0.790+/-0.010
  [INFLATED bal 1:1] AUC=0.910+/-0.004  AUPR=0.923+/-0.003
  [INFLATED imb 1:5] AUC=0.909+/-0.003  AUPR=0.781+/-0.011


In [15]:
import math

class DenseGTLayer(nn.Module):
    """Graph Transformer layer using dense N×N adjacency masking (no DGL needed).
    Ported from GCGB/graph_transformer_layer.py. Architecture:
      MultiHeadAttention (Q,K,V with scaled dot-product, graph-masked)
      + residual + LayerNorm + FFN + residual + LayerNorm.
    """
    def __init__(self, in_dim, out_dim, n_heads, dropout=0.0):
        super().__init__()
        assert out_dim % n_heads == 0
        self.n_heads = n_heads; self.d_head = out_dim // n_heads
        self.Q = nn.Linear(in_dim, out_dim); self.K = nn.Linear(in_dim, out_dim)
        self.V = nn.Linear(in_dim, out_dim); self.O = nn.Linear(out_dim, out_dim)
        self.FFN1 = nn.Linear(out_dim, out_dim*2); self.FFN2 = nn.Linear(out_dim*2, out_dim)
        self.ln1  = nn.LayerNorm(out_dim);          self.ln2  = nn.LayerNorm(out_dim)
        self.drop = nn.Dropout(dropout)
        self.res_proj = nn.Linear(in_dim, out_dim, bias=False) if in_dim != out_dim else None
        for m in [self.Q,self.K,self.V,self.O,self.FFN1,self.FFN2]:
            nn.init.xavier_uniform_(m.weight)

    def forward(self, h, adj):
        """h: (N, in_dim), adj: (N, N) binary adjacency."""
        N = h.size(0); H, D = self.n_heads, self.d_head
        Q = self.Q(h).view(N,H,D).permute(1,0,2)   # (H,N,D)
        K = self.K(h).view(N,H,D).permute(1,0,2)
        V = self.V(h).view(N,H,D).permute(1,0,2)
        score = torch.bmm(Q, K.transpose(1,2)) / math.sqrt(D)  # (H,N,N)
        score = score.masked_fill((adj==0).unsqueeze(0), -9e15)
        alpha = self.drop(torch.softmax(score, dim=-1))          # (H,N,N)
        out   = torch.bmm(alpha, V).permute(1,0,2).reshape(N, H*D)
        out   = self.O(out)
        h_res = h if self.res_proj is None else self.res_proj(h)
        out   = self.ln1(h_res + out)
        return self.ln2(out + self.FFN2(F.relu(self.FFN1(out))))

class SimGraphTransformer(nn.Module):
    """Stack of DenseGTLayers for a single-type similarity graph.
    Node features = rows of similarity matrix (in_dim = n_nodes_of_that_type).
    Returns list of per-layer (N, out_dim) representations.
    """
    def __init__(self, in_dim, hidden_dim, out_dim, n_layers, n_heads, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        dims   = [hidden_dim]*(n_layers-1) + [out_dim]
        in_dims= [hidden_dim] + [hidden_dim]*(n_layers-1)
        self.layers = nn.ModuleList([
            DenseGTLayer(in_dims[i], dims[i], n_heads, dropout) for i in range(n_layers)])

    def forward(self, features, adj):
        """features: (N, in_dim), adj: (N, N). Returns list of (N, d) per layer."""
        h = F.relu(self.input_proj(features))
        reps = []
        for layer in self.layers:
            h = layer(h, adj); reps.append(h)
        return reps

class Multi_Head_Self_ATT(nn.Module):
    """Per-node fusion of two same-dim views via multi-head attention.
    Source: GCGB/model.py:22-57.
    Query=mean(x1,x2), Keys=linear([x1,x2]), Values=[x1,x2].
    Output: weighted combination of x1 and x2 for each node.
    """
    def __init__(self, n_heads, in_channels, out_channels):
        super().__init__()
        assert out_channels % n_heads == 0
        self.n_heads = n_heads; self.out_channels = out_channels
        self.k_lin = nn.Linear(in_channels, out_channels)
        nn.init.xavier_uniform_(self.k_lin.weight)
        nn.init.constant_(self.k_lin.bias, 0.0)

    def forward(self, x1, x2):
        """x1, x2: (N, in_channels). Returns (N, out_channels)."""
        x = torch.stack([x1,x2], dim=1)   # (N,2,in_ch)
        N, M, _ = x.shape; H, D = self.n_heads, self.out_channels//self.n_heads
        q = torch.mean(x,dim=1).view(N,1,H,D)   # (N,1,H,D)
        k = self.k_lin(x).view(N,M,H,D)          # (N,2,H,D)
        v = x.view(N,M,H,D)
        q=q.permute(0,2,1,3); k=k.permute(0,2,3,1); v=v.permute(0,2,1,3)
        alpha = F.softmax(torch.matmul(q,k)/math.sqrt(D), dim=-1)  # (N,H,1,2)
        o     = torch.matmul(alpha, v)              # (N,H,1,D)
        return o.permute(0,2,1,3).reshape(N, H*D)   # (N, out_channels)

print("DenseGTLayer, SimGraphTransformer, Multi_Head_Self_ATT defined.")

DenseGTLayer, SimGraphTransformer, Multi_Head_Self_ATT defined.


In [16]:
class MPHAMv7(nn.Module):
    """EXP-F: Dual-stream encoder (GCGB architecture adapted).

    Stream A (sim):  SimGraphTransformer on drug-drug and disease-disease
                     k-NN similarity graphs. Features = similarity matrix rows.
    Stream B (ass):  MPHAMLayer (GAT) on meta-path adjacencies (decoupled H0).
    Fusion:          Multi_Head_Self_ATT(sim_emb, ass_emb) per entity type.
    Decoder:         RotatE + mul 4-way (same as MPHAMv5).

    sim captures content-based proximity; ass captures relational meta-paths.
    Fusing them gives each entity both views at inference.
    """
    def __init__(self, cfg, in_dim_meta, n_distances):
        super().__init__(); self.cfg = cfg
        nR, nD = cfg.n_drug, cfg.n_dis; d = cfg.emb_dim
        # Association stream (one MPHAM layer)
        self.mpham_layer = MPHAMLayer(in_dim_meta, d, n_distances, cfg)
        # Similarity stream (separate GT per entity type)
        # input feature dim = number of entities of that type (sim matrix row)
        self.gt_drug = SimGraphTransformer(nR, d, d, cfg.gt_layers, cfg.gt_heads, cfg.dropout)
        self.gt_dis  = SimGraphTransformer(nD, d, d, cfg.gt_layers, cfg.gt_heads, cfg.dropout)
        # Per-entity fusion
        self.drug_fusion = Multi_Head_Self_ATT(cfg.gt_heads, d, d)
        self.dis_fusion  = Multi_Head_Self_ATT(cfg.gt_heads, d, d)
        # 4-way RotatE+mul decoder
        self.predictor = nn.Sequential(
            nn.Linear(4*d, cfg.predictor_hidden), nn.ReLU(),
            nn.Dropout(cfg.dropout), nn.Linear(cfg.predictor_hidden, 1))

    def encode(self, H0, adjs, S_rr_t, S_dd_t):
        """
        H0:     (N_total, N_total) decoupled feature matrix
        adjs:   meta-path adjacencies dict
        S_rr_t: (nR, nR) drug similarity adjacency tensor (binary, on device)
        S_dd_t: (nD, nD) disease similarity adjacency tensor
        Returns: h_drug_fused (nR,d), h_dis_fused (nD,d)
        """
        nR, nD = self.cfg.n_drug, self.cfg.n_dis
        # Association stream
        H_ass = self.mpham_layer(H0, adjs)
        h_drug_ass = H_ass[:nR]; h_dis_ass = H_ass[nR:nR+nD]
        # Similarity stream (feature = similarity row = S_rr/S_dd row)
        l_drug_sim = self.gt_drug(S_rr_t, S_rr_t)   # features=S_rr, adj=S_rr
        l_dis_sim  = self.gt_dis (S_dd_t, S_dd_t)
        h_drug_sim = l_drug_sim[-1]; h_dis_sim = l_dis_sim[-1]
        # Fuse
        h_drug_fused = self.drug_fusion(h_drug_sim, h_drug_ass)
        h_dis_fused  = self.dis_fusion (h_dis_sim,  h_dis_ass)
        return h_drug_fused, h_dis_fused

    def score_direct(self, h_d, h_di):
        mul=h_d*h_di; rot=rotate_operator(h_d,h_di)
        return self.predictor(torch.cat([h_d,h_di,mul,rot],dim=-1)).squeeze(-1)

    def score(self, h_drug_fused, h_dis_fused, drug_idx, dis_idx):
        return self.score_direct(h_drug_fused[drug_idx], h_dis_fused[dis_idx])

print("MPHAMv7 (dual-stream: sim GT + ass MPHAM + fusion + RotatE) defined.")

MPHAMv7 (dual-stream: sim GT + ass MPHAM + fusion + RotatE) defined.


In [17]:
@torch.no_grad()
def evaluate_v7(cfg, model, H0, adjs, S_rr_t, S_dd_t, test_pos, test_neg, device):
    model.eval()
    hdr, hdi = model.encode(H0, adjs, S_rr_t, S_dd_t)
    pairs = np.concatenate([test_pos, test_neg], 0)
    y = np.concatenate([np.ones(len(test_pos)), np.zeros(len(test_neg))])
    di = torch.from_numpy(pairs[:,0]).to(device)
    si = torch.from_numpy(pairs[:,1]).to(device)
    prob = torch.sigmoid(model.score(hdr, hdi, di, si)).cpu().numpy()
    return _metrics(y, prob, cfg.score_threshold)

def run_fold_v7(cfg, ds, train_pos, test_pos, device, rng, val_frac=0.1):
    """EXP-F: MPHAMv7 dual-stream + bidirectional hard neg + optional sim CL."""
    nR, nD = cfg.n_drug, cfg.n_dis
    pos_set = set(map(int, ds.positives[:,0]*nD + ds.positives[:,1]))
    n_val = max(1, int(len(train_pos)*val_frac))
    vidx  = rng.choice(len(train_pos), n_val, replace=False)
    vmask = np.ones(len(train_pos), bool); vmask[vidx] = False
    val_pos=train_pos[vidx]; tr_pos=train_pos[vmask]

    A_rd_tr = mask_test_edges(ds.A_rd, test_pos)
    S_rr, S_dd, S_pp = ds.build_similarities(A_rd_tr)
    H0_np   = build_H0_decoupled(A_rd_tr, ds.A_rp, ds.A_pd, S_rr, S_dd, S_pp)
    adjs_np = build_metapath_adjs(cfg, A_rd_tr, ds.A_rp, ds.A_pd, S_rr, S_dd, S_pp)
    H0   = torch.from_numpy(H0_np).to(device)
    adjs = _adjs_to_torch(adjs_np, device)
    # sim adjacency tensors for GT stream (binarized)
    S_rr_t = torch.from_numpy((S_rr > 0).astype(np.float32)).to(device)
    S_dd_t = torch.from_numpy((S_dd > 0).astype(np.float32)).to(device)

    model = MPHAMv7(cfg, in_dim_meta=H0.shape[1], n_distances=len(adjs)).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    val_neg   = sample_negatives(pos_set, len(val_pos),    nR, nD, rng)
    test_neg1 = sample_negatives(pos_set, len(test_pos),   nR, nD, rng)
    test_neg5 = sample_negatives(pos_set, len(test_pos)*5, nR, nD, rng)
    best_val_auc=-1; best_test_auc=-1; patience=0
    best_state_val  = {k:v.clone() for k,v in model.state_dict().items()}
    best_state_test = {k:v.clone() for k,v in model.state_dict().items()}

    for epoch in range(cfg.epochs):
        model.train()
        if epoch==0 or cfg.resample_neg_each_epoch:
            tr_neg = sample_negatives(pos_set, len(tr_pos)*cfg.neg_ratio, nR, nD, rng)
        pos_di=torch.from_numpy(tr_pos[:,0]).to(device); pos_si=torch.from_numpy(tr_pos[:,1]).to(device)
        neg_di=torch.from_numpy(tr_neg[:,0]).to(device); neg_si=torch.from_numpy(tr_neg[:,1]).to(device)

        opt.zero_grad()
        hdr, hdi = model.encode(H0, adjs, S_rr_t, S_dd_t)
        h_dp=hdr[pos_di]; h_dip=hdi[pos_si]
        h_dn=hdr[neg_di]; h_din=hdi[neg_si]

        loss = F.binary_cross_entropy_with_logits(
            torch.cat([model.score_direct(h_dp,h_dip), model.score_direct(h_dn,h_din)]),
            torch.cat([torch.ones(len(pos_di),device=device), torch.zeros(len(neg_di),device=device)]))

        n_mix=min(len(h_dp),len(h_dn))
        sr=torch.mean(h_dp[:n_mix]*h_dn[:n_mix]).detach()
        h_drug_hard=sr*h_dp[:n_mix]+(1-sr)*h_dn[:n_mix]
        sd=torch.mean(h_dip[:n_mix]*h_din[:n_mix]).detach()
        h_dis_hard=sd*h_dip[:n_mix]+(1-sd)*h_din[:n_mix]
        loss = loss + cfg.hard_neg_weight * F.binary_cross_entropy_with_logits(
            model.score_direct(h_drug_hard, h_dis_hard), torch.zeros(n_mix, device=device))

        if cfg.sim_cl_weight > 0:
            loss = loss + cfg.sim_cl_weight * (
                sim_contrastive_loss(hdr, S_rr, cfg.sim_cl_temp) +
                sim_contrastive_loss(hdi, S_dd, cfg.sim_cl_temp))

        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()

        vauc,*_=evaluate_v7(cfg,model,H0,adjs,S_rr_t,S_dd_t,val_pos,val_neg,device)
        if vauc>best_val_auc:
            best_val_auc=vauc; patience=0
            best_state_val={k:v.clone() for k,v in model.state_dict().items()}
        else:
            patience+=1
            if patience>=cfg.early_stop_patience: break
        tauc,*_=evaluate_v7(cfg,model,H0,adjs,S_rr_t,S_dd_t,test_pos,test_neg1,device)
        if tauc>best_test_auc:
            best_test_auc=tauc
            best_state_test={k:v.clone() for k,v in model.state_dict().items()}

    def _eb(state):
        model.load_state_dict(state)
        return (evaluate_v7(cfg,model,H0,adjs,S_rr_t,S_dd_t,test_pos,test_neg1,device),
                evaluate_v7(cfg,model,H0,adjs,S_rr_t,S_dd_t,test_pos,test_neg5,device))
    h_bal,h_imb=_eb(best_state_val); i_bal,i_imb=_eb(best_state_test)
    return h_bal,h_imb,i_bal,i_imb

def cross_validate_v7(cfg, val_frac=0.1):
    """EXP-F: dual-stream encoder cross-validation."""
    torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
    device="cuda" if torch.cuda.is_available() else "cpu"
    print(f"[EXP-F] device={device}  emb={cfg.emb_dim}  attn={cfg.node_attn}"
          f"  topk={cfg.metapath_topk}  gt_layers={cfg.gt_layers}"
          f"  gt_heads={cfg.gt_heads}  ep={cfg.epochs}")
    ds=BDataset(cfg); pos=ds.positives
    rng=np.random.default_rng(cfg.seed); perm=rng.permutation(len(pos))
    folds=np.array_split(perm,cfg.n_folds)
    hb=[]; hi=[]; ib=[]; ii=[]
    for f in range(cfg.n_folds):
        tidx=folds[f]; trinx=np.concatenate([folds[g] for g in range(cfg.n_folds) if g!=f])
        h_bal,h_imb,i_bal,i_imb=run_fold_v7(cfg,ds,pos[trinx],pos[tidx],device,rng,val_frac)
        print(f"  fold {f}: HONEST AUC={h_bal[0]:.4f} AUPR1={h_bal[1]:.4f}"
              f" AUPR5={h_imb[1]:.4f}  | INFLATED AUC={i_bal[0]:.4f}")
        hb.append(h_bal); hi.append(h_imb); ib.append(i_bal); ii.append(i_imb)
    print(); hbm,hbs=_agg(hb,"HONEST   bal 1:1"); him,his=_agg(hi,"HONEST   imb 1:5")
    ibm,ibs=_agg(ib,"INFLATED bal 1:1"); _agg(ii,"INFLATED imb 1:5")
    return (hbm,hbs),(him,his),(ibm,ibs)

print("evaluate_v7, run_fold_v7, cross_validate_v7 defined.")

evaluate_v7, run_fold_v7, cross_validate_v7 defined.


In [18]:
print("="*65)
print("EXP-F: Dual-Stream Encoder (sim GT + ass MPHAM + fusion + RotatE)")
print("Config: emb=128, topk=10, GAT, gt_layers=2, gt_heads=4, epochs=300")
print("="*65)
expf_results = cross_validate_v7(
    Config(node_attn="gat", emb_dim=256, predictor_hidden=256,
           metapath_topk=10, epochs=300, hard_neg_weight=0.5,
           gt_layers=2, gt_heads=4)
)

EXP-F: Dual-Stream Encoder (sim GT + ass MPHAM + fusion + RotatE)
Config: emb=128, topk=10, GAT, gt_layers=2, gt_heads=4, epochs=300
[EXP-F] device=cuda  emb=256  attn=gat  topk=10  gt_layers=2  gt_heads=4  ep=300
  fold 0: HONEST AUC=0.9099 AUPR1=0.9228 AUPR5=0.7916  | INFLATED AUC=0.9115
  fold 1: HONEST AUC=0.8993 AUPR1=0.9162 AUPR5=0.7769  | INFLATED AUC=0.9014
  fold 2: HONEST AUC=0.9094 AUPR1=0.9242 AUPR5=0.8038  | INFLATED AUC=0.9108
  fold 3: HONEST AUC=0.9138 AUPR1=0.9243 AUPR5=0.7845  | INFLATED AUC=0.9138
  fold 4: HONEST AUC=0.9100 AUPR1=0.9244 AUPR5=0.7867  | INFLATED AUC=0.9105

  [HONEST   bal 1:1] AUC=0.908+/-0.005  AUPR=0.922+/-0.003
  [HONEST   imb 1:5] AUC=0.909+/-0.004  AUPR=0.789+/-0.009
  [INFLATED bal 1:1] AUC=0.910+/-0.004  AUPR=0.923+/-0.003
  [INFLATED imb 1:5] AUC=0.910+/-0.004  AUPR=0.792+/-0.009
